# Unsupervised Machine Learning with Scikit-Learn

This notebook provides complete, hands-on implementations and evaluations of core **Unsupervised Learning** paradigms using built-in **scikit-learn** datasets:

1. **Hierarchical (Agglomerative) Clustering** — *Iris Dataset (Dendrogram & Ward Linkage)*
2. **Density-Based Clustering (DBSCAN)** — *Non-linear Crescent Dataset (Outlier Detection & Arbitrary Shapes)*
3. **K-Means Clustering** — *Iris Dataset (Elbow Method, Silhouette Analysis, Centroids)*
4. **Association Rule Mining** — *Market Basket Analysis (Apriori, Support, Confidence, Lift)*
5. **Dimensionality Reduction** — *Digits Dataset (Linear TruncatedSVD vs. Non-linear t-SNE)*
6. **Principal Component Analysis (PCA)** — *Breast Cancer Dataset (Scree Plot, Loadings, 2D/3D Projections, Reconstruction Loss)*
7. **Comparative Summary & Architecture Guide**

---

### Pedagogical Template (8-Step Architecture):
To maximize educational clarity for students and serve as a structured revision guide, every section adheres to the uniform 8-step workflow (adapted for unsupervised workflows):
1. **Importing** — Libraries, transformer classes, and unsupervised evaluation metrics.
2. **Loading Dataset** — Data loading, shape inspection, and exploratory review.
3. **Preprocessing / Scaling** — Feature standardization (`StandardScaler`), transaction encoding, or optimal parameter diagnostics.
4. **Train / Model Setup** — Algorithm instantiation and hyperparameter configuration.
5. **Fit** — Fitting the unsupervised estimator on the feature space (`.fit()`).
6. **Predict / Transform** — Cluster assignment (`.predict()` / `.labels_`) or dimensional projection (`.transform()`).
7. **Evaluation** — Unsupervised validation metrics (Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz, Inertia, Lift, Explained Variance).
8. **Visualization** — Diagnostic plots (Dendrograms, Silhouette plots, Centroid scatter, Scree plots, Loadings heatmaps, 3D projections).


---
# 1. Hierarchical (Agglomerative) Clustering

Hierarchical Clustering is a bottom-up clustering paradigm where each data point starts in its own individual cluster. Successive pairs of clusters are iteratively merged based on a linkage criterion until all points belong to a single root cluster.
- **Ward's Linkage:** Minimizes the total within-cluster variance when merging two clusters.
- **Dendrogram:** A tree diagram recording the sequence and Euclidean distance of every cluster merge.

> 🍋 **Layman's Analogy ("Explain Like I'm 5"):**
> Imagine building a giant family tree in reverse. On Day 1, everyone in town is an individual. On Day 2, the closest twin brothers hold hands and form a pair. On Day 3, cousins join in to form family circles. Soon, whole neighborhoods merge until the whole town connects into one giant tree. The **Dendrogram** is simply the family tree diagram! Drawing a horizontal line across it lets you choose whether you want 3 big clans or 10 smaller families.


### Step 1: Importing

> 💡 *Beginner Tip:* We import only the exact clustering classes, dendrogram tools, and evaluation metrics needed for this recipe.
**Key Functions Explained:**
- `AgglomerativeClustering`: Bottom-up hierarchical clustering estimator.
- `linkage`, `dendrogram`: Scipy hierarchical clustering utilities for tree matrix computation and dendrogram plotting.
- `silhouette_score`, `davies_bouldin_score`, `calinski_harabasz_score`: Standard unsupervised clustering validation metrics.


In [ ]:
# 1. Importing libraries for Hierarchical Clustering
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage

# Visualization styles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11



### Step 2: Loading Dataset

> 💡 *Beginner Tip:* We load the raw data into a DataFrame. Notice there are NO labels for training—the algorithm must discover patterns purely from feature measurements.
Loading Iris dataset (150 samples, 4 features).

[`Iris Dataset`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html)


In [ ]:
# 2. Loading Dataset
iris_agg = load_iris(as_frame=True)
X_agg = iris_agg.data

print(f"Iris Dataset Shape: {X_agg.shape}")
print("Features:", list(X_agg.columns))
display(X_agg.head(3))


### Step 3: Preprocessing & Scaling

> 💡 *Beginner Tip:* Distance-based algorithms (like clustering and PCA) are extremely sensitive to scale! We standardize numbers so huge measurements don't bully smaller ones.
**Why scaling is required:**
Hierarchical clustering calculates Euclidean distances between samples. If one feature has a much larger scale, it will artificially dominate the distances. We use `StandardScaler` to ensure all features contribute equally.


In [ ]:
# 3. Preprocessing: Feature Standardization
scaler_agg = StandardScaler()
X_agg_scaled = scaler_agg.fit_transform(X_agg)

print("Scaled feature means (approx 0):", X_agg_scaled.mean(axis=0).round(2))
print("Scaled feature standard deviations:", X_agg_scaled.std(axis=0).round(2))


### Step 4: Train (Model Initialization & Dendrogram Diagnostics)

> 💡 *Beginner Tip:* We initialize our clustering tool and set our preferences, such as how many clusters ($k$) or what distance threshold to use.
**Key Parameters Explained:**
- `n_clusters=3`: Target number of clusters to form.
- `linkage='ward'`: Minimizes the variance of clusters being merged.


In [ ]:
# 4. Train: Model Initialization
agg_cluster = AgglomerativeClustering(n_clusters=3, linkage='ward')
print("Model initialized:", agg_cluster)


### Step 5: Fit

> 💡 *Beginner Tip:* The computer groups the points together based on closeness, density, or variance.
Fitting the agglomerative hierarchy on scaled data.


In [ ]:
# 5. Fit: Model Training
agg_cluster.fit(X_agg_scaled)
print("Agglomerative Clustering fit complete!")


### Step 6: Predict (Cluster Label Assignment)

> 💡 *Beginner Tip:* We assign every data point its official cluster tag (0, 1, 2... or -1 for noise).
Extracting the assigned cluster labels (0, 1, or 2) for each observation.


In [ ]:
# 6. Predict: Cluster Assignment
agg_labels = agg_cluster.labels_
print(f"Cluster label counts: {pd.Series(agg_labels).value_counts().to_dict()}")
print("First 10 assigned cluster labels:", agg_labels[:10])


### Step 7: Evaluation

> 💡 *Beginner Tip:* Since we don't have true answer sheets, we grade clusters using intrinsic geometric quality: Silhouette score measures separation and cohesion.
**Unsupervised Evaluation Metrics Explained:**
- **Silhouette Score:** Measures cohesion within clusters vs. separation from neighbors (range: $-1$ to $+1$; higher is better).
- **Davies-Bouldin Index (DBI):** Average similarity between each cluster and its most similar one (lower is better; 0 is perfect).
- **Calinski-Harabasz Index:** Ratio of between-cluster dispersion to within-cluster dispersion (higher is better).


In [ ]:
# 7. Evaluation
sil_agg = silhouette_score(X_agg_scaled, agg_labels)
db_agg = davies_bouldin_score(X_agg_scaled, agg_labels)
ch_agg = calinski_harabasz_score(X_agg_scaled, agg_labels)

print("=" * 60)
print("HIERARCHICAL CLUSTERING EVALUATION REPORT (k=3)")
print("=" * 60)
print(f"Silhouette Score (higher is better, max 1.0): {sil_agg:.4f}")
print(f"Davies-Bouldin Index (lower is better):      {db_agg:.4f}")
print(f"Calinski-Harabasz Score (higher is better):  {ch_agg:.2f}")
print("=" * 60)


### Step 8: Visualization

> 💡 *Beginner Tip:* We plot colorful 2D maps, decision boundaries, or tree dendrograms to visually inspect how well the groups separated.
**Visualizations:**
1. **Dendrogram:** Visual representation of the hierarchical merge tree with horizontal cut threshold.
2. **2D Cluster Scatter Plot:** Visualizing clusters on petal dimensions.


In [ ]:
# 8. Visualization: Dendrogram & Cluster Scatter Plot
# Plot 1: Dendrogram
linked_mat = linkage(X_agg_scaled, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(
    linked_mat,
    truncate_mode='lastp',
    p=25,
    leaf_rotation=45.,
    leaf_font_size=10.,
    show_contracted=True
)
plt.title("Hierarchical Clustering Dendrogram (Ward Linkage)", fontsize=14, fontweight="bold")
plt.xlabel("Cluster Size / Sample Index", fontsize=11)
plt.ylabel("Ward Linkage Distance", fontsize=11)
plt.axhline(y=10.0, color='r', linestyle='--', label='Cut Threshold (k=3 clusters)')
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: 2D Cluster Scatter
plt.figure(figsize=(9, 5))
plt.scatter(
    X_agg.iloc[:, 2], X_agg.iloc[:, 3],
    c=agg_labels, cmap='viridis', s=60, edgecolors='k', alpha=0.85
)
plt.title(f"Agglomerative Clusters on Petal Dimensions (k=3, Silhouette = {sil_agg:.3f})", fontsize=13, fontweight="bold")
plt.xlabel("Petal Length (cm)", fontsize=11)
plt.ylabel("Petal Width (cm)", fontsize=11)
plt.tight_layout()
plt.show()


---
# 2. Density-Based Clustering (DBSCAN)

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) groups points that are closely packed together (high density) while marking points in low-density regions as **noise / outliers** (assigned label `-1`).

**Two Core Parameters:**
- `eps` ($\epsilon$): Maximum neighborhood radius around a point.
- `min_samples`: Minimum number of points within radius $\epsilon$ required to form a core point.

> 🍋 **Layman's Analogy ("Explain Like I'm 5"):**
> Imagine looking at Earth from an airplane at night. Where streetlights and houses are packed tightly together, that is a bustling city (**Core Cluster**). Along the edges, quieter suburbs touch the city (**Border Points**). A lone lighthouse out in the middle of a dark, empty ocean belongs to no city at all—it is an outlier (**Noise Point, label -1**). Unlike other algorithms, DBSCAN doesn't force round circles and automatically spots weirdly shaped islands while throwing away lonely noise!


### Step 1: Importing

> 💡 *Beginner Tip:* We import only the exact clustering classes, dendrogram tools, and evaluation metrics needed for this recipe.
Imports for DBSCAN.


In [ ]:
# 1. Importing libraries for DBSCAN
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score



### Step 2: Loading Dataset

> 💡 *Beginner Tip:* We load the raw data into a DataFrame. Notice there are NO labels for training—the algorithm must discover patterns purely from feature measurements.
Generating a non-linear, crescent-shaped two-moons dataset (`make_moons`) to demonstrate DBSCAN's ability to discover arbitrary non-convex geometries.

[`make_moons Dataset`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html)


In [ ]:
# 2. Loading Dataset (Non-linear Crescent Moons)
X_moons, y_moons = make_moons(n_samples=400, noise=0.08, random_state=42)
print(f"Generated Synthetic Moons Dataset: {X_moons.shape} (400 samples, 2 features)")
df_moons = pd.DataFrame(X_moons, columns=['Feature_1', 'Feature_2'])
display(df_moons.head(3))


### Step 3: Preprocessing & Scaling

> 💡 *Beginner Tip:* Distance-based algorithms (like clustering and PCA) are extremely sensitive to scale! We standardize numbers so huge measurements don't bully smaller ones.
Standardizing spatial coordinates.


In [ ]:
# 3. Preprocessing: Feature Scaling
scaler_db = StandardScaler()
X_moons_scaled = scaler_db.fit_transform(X_moons)
print("Features scaled successfully.")


### Step 4: Train (Model Initialization)

> 💡 *Beginner Tip:* We initialize our clustering tool and set our preferences, such as how many clusters ($k$) or what distance threshold to use.
**Key Hyperparameters:**
- `eps=0.25`: Maximum distance between two points to be considered neighbors.
- `min_samples=5`: Core point neighbor requirement.


In [ ]:
# 4. Train: Model Initialization
dbscan = DBSCAN(eps=0.25, min_samples=5)
print("Model initialized:", dbscan)


### Step 5: Fit

> 💡 *Beginner Tip:* The computer groups the points together based on closeness, density, or variance.
Fitting density components across the point cloud.


In [ ]:
# 5. Fit: Model Training
dbscan.fit(X_moons_scaled)
print("DBSCAN clustering fit complete!")


### Step 6: Predict (Extracting Cluster Labels & Outliers)

> 💡 *Beginner Tip:* We assign every data point its official cluster tag (0, 1, 2... or -1 for noise).
Extracting cluster labels. Points labeled `-1` are detected as noise/outliers.


In [ ]:
# 6. Predict: Cluster & Outlier Assignment
dbscan_labels = dbscan.labels_

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_db = list(dbscan_labels).count(-1)

print(f"Clusters Discovered:     {n_clusters_db}")
print(f"Noise Points Detected:   {n_noise_db} ({n_noise_db / len(X_moons) * 100:.1f}%)")


### Step 7: Evaluation

> 💡 *Beginner Tip:* Since we don't have true answer sheets, we grade clusters using intrinsic geometric quality: Silhouette score measures separation and cohesion.
Evaluating silhouette score on non-noise core points.


In [ ]:
# 7. Evaluation
core_mask = dbscan_labels != -1
sil_dbscan = silhouette_score(X_moons_scaled[core_mask], dbscan_labels[core_mask])

print("=" * 60)
print("DBSCAN EVALUATION REPORT")
print("=" * 60)
print(f"Number of Clusters Formed:         {n_clusters_db}")
print(f"Total Noise Outliers:              {n_noise_db}")
print(f"Core Silhouette Score:             {sil_dbscan:.4f}")
print("=" * 60)


### Step 8: Visualization

> 💡 *Beginner Tip:* We plot colorful 2D maps, decision boundaries, or tree dendrograms to visually inspect how well the groups separated.
Plotting clusters and distinctly marking noise points in black.


In [ ]:
# 8. Visualization: DBSCAN Non-linear Shapes and Noise Points
plt.figure(figsize=(9, 6))

unique_labels = set(dbscan_labels)
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]

for k, col in zip(unique_labels, colors):
    if k == -1:
        col = [0, 0, 0, 1]
        label_text = "Noise / Outlier (-1)"
    else:
        label_text = f"Cluster {k}"

    class_member_mask = (dbscan_labels == k)
    xy = X_moons_scaled[class_member_mask]
    plt.scatter(xy[:, 0], xy[:, 1], color=tuple(col), edgecolors='k', s=55, alpha=0.85, label=label_text)

plt.title("DBSCAN: Arbitrary Non-Linear Shape Discovery & Outlier Detection", fontsize=13, fontweight="bold")
plt.xlabel("Scaled Feature 1", fontsize=11)
plt.ylabel("Scaled Feature 2", fontsize=11)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()


---
# 3. K-Means Clustering

K-Means partitions $N$ observations into $K$ clusters by minimizing the within-cluster sum of squares (**Inertia / WCSS**):
$$J = \sum_{k=1}^K \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

Where $\mu_k$ is the centroid (mean vector) of cluster $C_k$.

> 🍋 **Layman's Analogy ("Explain Like I'm 5"):**
> You are hired by a university to place **3 Wi-Fi routers** in a large student hall:
> 1. You drop 3 routers at random locations.
> 2. Each student connects to whichever router is closest to them.
> 3. At the end of the day, you pick up each router and move it to the exact middle (centroid) of the students connected to it.
> 4. You repeat this dance until the routers stop moving! That is K-Means.


### Step 1: Importing

> 💡 *Beginner Tip:* We import only the exact clustering classes, dendrogram tools, and evaluation metrics needed for this recipe.
Imports for K-Means Clustering.


In [ ]:
# 1. Importing libraries for K-Means Clustering
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score
)



### Step 2: Loading Dataset

> 💡 *Beginner Tip:* We load the raw data into a DataFrame. Notice there are NO labels for training—the algorithm must discover patterns purely from feature measurements.
Loading Iris dataset.

[`Iris Dataset`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html)


In [ ]:
# 2. Loading Dataset
iris_km = load_iris(as_frame=True)
X_km = iris_km.data
print(f"Iris Dataset Shape: {X_km.shape}")
display(X_km.head(3))


### Step 3: Preprocessing, Scaling & Optimal K Diagnostics

> 💡 *Beginner Tip:* Distance-based algorithms (like clustering and PCA) are extremely sensitive to scale! We standardize numbers so huge measurements don't bully smaller ones.
**Key Diagnostics to Determine $K$:**
1. **The Elbow Method:** Plots Inertia vs. $K$. The bend ("elbow") indicates diminishing returns.
2. **Silhouette Analysis:** Computes average silhouette score for $K \in [2, 8]$.


In [ ]:
# 3. Preprocessing, Scaling & Optimal K Diagnostics
scaler_km = StandardScaler()
X_km_scaled = scaler_km.fit_transform(X_km)

k_range = range(2, 9)
inertias = []
sil_scores = []

for k in k_range:
    km_test = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km_test.fit(X_km_scaled)
    inertias.append(km_test.inertia_)
    sil_scores.append(silhouette_score(X_km_scaled, km_test.labels_))

# Visualize Elbow and Silhouette curves side-by-side
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(k_range, inertias, marker='o', color='#1f77b4', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', label='Elbow Point (k=3)')
axes[0].set_title("The Elbow Method (Inertia vs. k)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Number of Clusters (k)", fontsize=11)
axes[0].set_ylabel("Inertia (WCSS)", fontsize=11)
axes[0].legend()

axes[1].plot(k_range, sil_scores, marker='s', color='#2ca02c', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', label='Optimal k=3')
axes[1].set_title("Silhouette Score vs. k", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Number of Clusters (k)", fontsize=11)
axes[1].set_ylabel("Mean Silhouette Score", fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()


### Step 4: Train (Model Initialization)

> 💡 *Beginner Tip:* We initialize our clustering tool and set our preferences, such as how many clusters ($k$) or what distance threshold to use.
**Key Hyperparameters:**
- `n_clusters=3`: Target number of clusters (guided by the Elbow/Silhouette plots).
- `init='k-means++'`: Smart centroid initialization that spaces out initial centers, dramatically speeding convergence and avoiding bad local minima.
- `n_init=10`: Runs algorithm 10 times with different centroid seeds and picks the one with lowest inertia.


In [ ]:
# 4. Train: Model Initialization
kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
print("Model initialized:", kmeans)


### Step 5: Fit

> 💡 *Beginner Tip:* The computer groups the points together based on closeness, density, or variance.
Fitting centroids via iterative Lloyd-Forgy expectation-maximization.


In [ ]:
# 5. Fit: Model Training
kmeans.fit(X_km_scaled)
print(f"K-Means fit complete! Final Inertia (WCSS): {kmeans.inertia_:.2f}")


### Step 6: Predict

> 💡 *Beginner Tip:* We assign every data point its official cluster tag (0, 1, 2... or -1 for noise).
Generating cluster labels for each sample.


In [ ]:
# 6. Predict: Cluster Label Assignment
km_labels = kmeans.predict(X_km_scaled)
print("Cluster Distribution:", pd.Series(km_labels).value_counts().to_dict())


### Step 7: Evaluation

> 💡 *Beginner Tip:* Since we don't have true answer sheets, we grade clusters using intrinsic geometric quality: Silhouette score measures separation and cohesion.
Evaluating Inertia, Silhouette Score, Davies-Bouldin Index, and Calinski-Harabasz Score.


In [ ]:
# 7. Evaluation
km_sil = silhouette_score(X_km_scaled, km_labels)
km_db = davies_bouldin_score(X_km_scaled, km_labels)
km_ch = calinski_harabasz_score(X_km_scaled, km_labels)

print("=" * 60)
print("K-MEANS CLUSTERING EVALUATION REPORT (k=3)")
print("=" * 60)
print(f"Final Inertia (WCSS):             {kmeans.inertia_:.2f}")
print(f"Silhouette Score (higher=better):  {km_sil:.4f}")
print(f"Davies-Bouldin Index (lower=better): {km_db:.4f}")
print(f"Calinski-Harabasz (higher=better): {km_ch:.2f}")
print("=" * 60)


### Step 8: Visualization

> 💡 *Beginner Tip:* We plot colorful 2D maps, decision boundaries, or tree dendrograms to visually inspect how well the groups separated.
**Visualizations:**
1. **Silhouette Sample Plot:** Visualizes silhouette coefficients for each individual sample across each cluster.
2. **2D Cluster Scatter Plot with Centroids:** Visualizes points colored by cluster with cluster centroids marked as red $\mathbf{X}$.


In [ ]:
# 8. Visualization: Silhouette Plot & Centroids Scatter
# Plot 1: Silhouette Sample Plot
sample_sil_values = silhouette_samples(X_km_scaled, km_labels)
fig, ax = plt.subplots(figsize=(9, 5))
y_lower = 10

for i in range(3):
    ith_vals = sample_sil_values[km_labels == i]
    ith_vals.sort()
    size_i = ith_vals.shape[0]
    y_upper = y_lower + size_i
    color = plt.cm.nipy_spectral(float(i) / 3)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_vals, facecolor=color, edgecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_i, f"Cluster {i}", fontsize=11, fontweight="bold")
    y_lower = y_upper + 10

ax.axvline(x=km_sil, color="red", linestyle="--", label=f"Average Score ({km_sil:.3f})")
ax.set_title("Silhouette Plot for K-Means (k=3) Clusters", fontsize=13, fontweight="bold")
ax.set_xlabel("Silhouette Coefficient Values", fontsize=11)
ax.set_ylabel("Cluster Label", fontsize=11)
ax.set_yticks([])
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Plot 2: 2D Cluster Scatter with Centroids
plt.figure(figsize=(9, 5))
plt.scatter(
    X_km_scaled[:, 2], X_km_scaled[:, 3],
    c=km_labels, cmap='viridis', s=65, edgecolors='k', alpha=0.85
)
centroids = kmeans.cluster_centers_
plt.scatter(
    centroids[:, 2], centroids[:, 3],
    marker='X', s=250, c='red', edgecolors='black', linewidths=2,
    label='Cluster Centroids'
)
plt.title("K-Means (k=3) Clusters with Learned Centroids", fontsize=14, fontweight="bold")
plt.xlabel("Petal Length (Standardized)", fontsize=11)
plt.ylabel("Petal Width (Standardized)", fontsize=11)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


---
# 4. Association Rule Mining (Apriori Algorithm)

Association Rule Mining discovers co-occurrence relationships and affinity patterns in transactional datasets:
$$\text{Antecedent } (X) \implies \text{Consequent } (Y)$$

**Key Metrics:**
- **Support:** $P(X \cap Y) = \frac{\text{Transactions with } X \text{ and } Y}{N}$
- **Confidence:** $P(Y|X) = \frac{\text{Support}(X \cap Y)}{\text{Support}(X)}$
- **Lift:** $\frac{\text{Support}(X \cap Y)}{\text{Support}(X) \cdot \text{Support}(Y)}$ (Lift > 1 implies positive correlation).
- **Leverage:** $\text{Support}(X \cap Y) - \text{Support}(X) \cdot \text{Support}(Y)$.

> 🍋 **Layman's Analogy ("Explain Like I'm 5"):**
> Supermarket detectives analyze grocery receipts to discover: *"What items are bought together?"*
> If someone buys burger patties and buns, they are 85% likely to also buy ketchup (**Confidence**). If people buy ketchup 20% of the time normally, but buy ketchup 80% of the time with burgers, burgers "lift" ketchup sales by 4x (**Lift = 4.0**). The store uses this trick to place ketchup right beside the meat counter!


### Step 1: Importing

> 💡 *Beginner Tip:* We import only the exact clustering classes, dendrogram tools, and evaluation metrics needed for this recipe.
Imports for Apriori and Association Rules via `mlxtend`.


In [ ]:
# 1. Importing libraries for Association Rule Mining
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules



### Step 2: Loading Dataset

> 💡 *Beginner Tip:* We load the raw data into a DataFrame. Notice there are NO labels for training—the algorithm must discover patterns purely from feature measurements.
Creating a realistic retail grocery market basket transaction list.

[`Retail Transactions Dataset`](https://rasbt.github.io/mlxtend/user_guide/preprocessing/TransactionEncoder/)


In [ ]:
# 2. Loading Dataset (Retail Market Basket)
transactions = [
    ['Milk', 'Bread', 'Butter'],
    ['Bread', 'Butter', 'Jam'],
    ['Milk', 'Bread', 'Butter', 'Jam'],
    ['Milk', 'Diapers', 'Beer', 'Eggs'],
    ['Milk', 'Bread', 'Diapers', 'Beer'],
    ['Bread', 'Butter', 'Diapers', 'Beer'],
    ['Milk', 'Bread', 'Butter', 'Eggs'],
    ['Beer', 'Diapers', 'Eggs'],
    ['Milk', 'Bread', 'Butter', 'Diapers', 'Beer'],
    ['Bread', 'Butter', 'Jam', 'Milk'],
    ['Milk', 'Diapers', 'Beer', 'Butter'],
    ['Bread', 'Eggs', 'Jam'],
    ['Milk', 'Bread', 'Eggs'],
    ['Diapers', 'Beer', 'Chips'],
    ['Milk', 'Bread', 'Butter', 'Beer', 'Diapers']
]

print(f"Total Transactions: {len(transactions)}")
print("Sample Transaction 1:", transactions[0])
print("Sample Transaction 4:", transactions[3])


### Step 3: Preprocessing (One-Hot Transaction Encoding)

> 💡 *Beginner Tip:* Distance-based algorithms (like clustering and PCA) are extremely sensitive to scale! We standardize numbers so huge measurements don't bully smaller ones.
Converting the raw list of item lists into a binary boolean one-hot matrix using `TransactionEncoder`.


In [ ]:
# 3. Preprocessing: Transaction Encoding
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_basket = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Encoded DataFrame Shape: {df_basket.shape} (15 transactions, {len(df_basket.columns)} unique items)")
display(df_basket.head(3))


### Step 4: Train / Setup (Frequent Itemset Mining Parameters)

> 💡 *Beginner Tip:* We initialize our clustering tool and set our preferences, such as how many clusters ($k$) or what distance threshold to use.
Configuring the `apriori` algorithm with `min_support=0.20` (items appearing in at least 20% of all carts).


In [ ]:
# 4. Train: Apriori Frequent Itemset Configuration
print("Configuring Apriori algorithm with minimum support threshold = 0.20")


### Step 5: Fit (Mining Frequent Itemsets)

> 💡 *Beginner Tip:* The computer groups the points together based on closeness, density, or variance.
Executing the Apriori pruning algorithm to identify frequent itemsets.


In [ ]:
# 5. Fit: Mining Frequent Itemsets
frequent_itemsets = apriori(df_basket, min_support=0.20, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False).reset_index(drop=True)

print(f"Frequent Itemsets Found: {len(frequent_itemsets)}")
display(frequent_itemsets.head(5))


### Step 6: Predict / Rule Derivation

> 💡 *Beginner Tip:* We assign every data point its official cluster tag (0, 1, 2... or -1 for noise).
Extracting association rules that meet a minimum confidence threshold of 0.60 (60%).


In [ ]:
# 6. Predict: Association Rule Extraction
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.60)
rules['rule'] = rules['antecedents'].apply(lambda x: ', '.join(list(x))) + " => " + rules['consequents'].apply(lambda x: ', '.join(list(x)))
rules_sorted = rules.sort_values(by='lift', ascending=False).reset_index(drop=True)

print(f"Total Rules Extracted: {len(rules_sorted)}")


### Step 7: Evaluation

> 💡 *Beginner Tip:* Since we don't have true answer sheets, we grade clusters using intrinsic geometric quality: Silhouette score measures separation and cohesion.
Evaluating top association rules by Support, Confidence, Lift, and Leverage.


In [ ]:
# 7. Evaluation: Top Association Rules Table
print("=" * 80)
print("TOP 10 ASSOCIATION RULES SORTED BY LIFT")
print("=" * 80)
display(rules_sorted[['rule', 'support', 'confidence', 'lift', 'leverage']].head(10).round(4))


### Step 8: Visualization

> 💡 *Beginner Tip:* We plot colorful 2D maps, decision boundaries, or tree dendrograms to visually inspect how well the groups separated.
**Visualizations:**
1. **Item Frequency Bar Plot:** Support of individual grocery items.
2. **Support vs. Confidence Scatter Plot:** Rule landscape with color and marker size proportional to Lift.


In [ ]:
# 8. Visualization: Item Support and Rule Landscape
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Item Frequency
item_support = df_basket.mean().sort_values(ascending=False)
axes[0].barh(item_support.index, item_support.values, color='#4575b4', edgecolor='black', alpha=0.85)
axes[0].set_title("Item Frequency (Individual Support)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Support (Fraction of Transactions)", fontsize=11)
axes[0].set_ylabel("Item", fontsize=11)

# Plot 2: Rules Scatter Plot
scatter_r = axes[1].scatter(
    rules['support'], rules['confidence'], c=rules['lift'], cmap='viridis',
    s=rules['lift'] * 50, edgecolors='black', alpha=0.85
)
cbar = plt.colorbar(scatter_r, ax=axes[1])
cbar.set_label("Lift Score", fontsize=11)
axes[1].set_title("Association Rules: Support vs. Confidence (Color/Size = Lift)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Support", fontsize=11)
axes[1].set_ylabel("Confidence", fontsize=11)

plt.tight_layout()
plt.show()


---
# 5. Dimensionality Reduction (Linear TruncatedSVD vs. Non-Linear t-SNE)

High-dimensional data suffers from the **Curse of Dimensionality** (sparsity, computational expense, and distance metric degradation).
- **Truncated SVD (Linear):** Performs matrix decomposition $X = U \Sigma V^T$ to preserve global variance linearly.
- **t-SNE (Non-Linear Manifold Learning):** Converts pairwise similarities into probabilities and minimizes KL divergence to reveal complex non-linear cluster structures in 2D/3D.

> 🍋 **Layman's Analogy ("Explain Like I'm 5"):**
> Imagine you have a 3D clay sculpture of a dragon with 64 tiny details:
> - **TruncatedSVD** is like shining a flashlight on the dragon and casting a 2D shadow on the wall. You lose 3D depth, but you capture the main outline quickly.
> - **t-SNE** is like carefully peeling the dragon's skin and flattening it out like a world map, making sure neighboring scales stay right next to each other on the flat table!


### Step 1: Importing

> 💡 *Beginner Tip:* We import only the exact clustering classes, dendrogram tools, and evaluation metrics needed for this recipe.
Imports for TruncatedSVD and t-SNE.


In [ ]:
# 1. Importing libraries for Dimensionality Reduction
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE



### Step 2: Loading Dataset

> 💡 *Beginner Tip:* We load the raw data into a DataFrame. Notice there are NO labels for training—the algorithm must discover patterns purely from feature measurements.
Loading handwritten Digits dataset (1797 images, 64 pixel features, classes 0 to 9).

[`Digits Dataset`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)


In [ ]:
# 2. Loading Dataset
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Digits Dataset Shape: {X_digits.shape} (64 pixel dimensions)")
print(f"Digit Classes: {np.unique(y_digits)}")


### Step 3: Preprocessing & Scaling

> 💡 *Beginner Tip:* Distance-based algorithms (like clustering and PCA) are extremely sensitive to scale! We standardize numbers so huge measurements don't bully smaller ones.
Standardizing 64 pixel features.


In [ ]:
# 3. Preprocessing: Feature Standardization
scaler_dr = StandardScaler()
X_digits_scaled = scaler_dr.fit_transform(X_digits)
print("64-dimensional features standardized.")


### Step 4: Train / Model Setup

> 💡 *Beginner Tip:* We initialize our clustering tool and set our preferences, such as how many clusters ($k$) or what distance threshold to use.
Initializing both Linear (TruncatedSVD) and Non-Linear (t-SNE) 2D projection models.


In [ ]:
# 4. Train: Model Initialization
svd_model = TruncatedSVD(n_components=2, random_state=42)
tsne_model = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')

print("Models initialized: TruncatedSVD (linear) and t-SNE (non-linear).")


### Step 5: Fit

> 💡 *Beginner Tip:* The computer groups the points together based on closeness, density, or variance.
Fitting models on the standardized digits data.


In [ ]:
# 5. Fit: Model Training
svd_model.fit(X_digits_scaled)
print("Truncated SVD fitted.")


### Step 6: Predict / Transform

> 💡 *Beginner Tip:* We assign every data point its official cluster tag (0, 1, 2... or -1 for noise).
Transforming 64-dimensional data into 2D coordinates.


In [ ]:
# 6. Predict / Transform: 2D Projection
X_svd_2d = svd_model.transform(X_digits_scaled)
X_tsne_2d = tsne_model.fit_transform(X_digits_scaled)

print(f"SVD 2D coordinates shape:  {X_svd_2d.shape}")
print(f"t-SNE 2D coordinates shape:{X_tsne_2d.shape}")


### Step 7: Evaluation

> 💡 *Beginner Tip:* Since we don't have true answer sheets, we grade clusters using intrinsic geometric quality: Silhouette score measures separation and cohesion.
Evaluating Explained Variance Ratio (SVD) and KL Divergence (t-SNE).


In [ ]:
# 7. Evaluation
var_svd = svd_model.explained_variance_ratio_.sum()
kl_tsne = tsne_model.kl_divergence_

print("=" * 60)
print("DIMENSIONALITY REDUCTION EVALUATION")
print("=" * 60)
print(f"Truncated SVD 2D Variance Explained: {var_svd * 100:.2f}%")
print(f"t-SNE Final KL Divergence:           {kl_tsne:.4f}")
print("=" * 60)


### Step 8: Visualization

> 💡 *Beginner Tip:* We plot colorful 2D maps, decision boundaries, or tree dendrograms to visually inspect how well the groups separated.
Side-by-side 2D scatter comparison showing how t-SNE dramatically unrolls the non-linear manifold and cleanly clusters all 10 digit classes.


In [ ]:
# 8. Visualization: TruncatedSVD vs. t-SNE 2D Projections
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: SVD Projection
scatter_s = axes[0].scatter(X_svd_2d[:, 0], X_svd_2d[:, 1], c=y_digits, cmap='tab10', s=30, alpha=0.75)
axes[0].set_title(f"Truncated SVD (Linear Projection, {var_svd*100:.1f}% Variance)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("SVD Component 1", fontsize=11)
axes[0].set_ylabel("SVD Component 2", fontsize=11)
plt.colorbar(scatter_s, ax=axes[0], label="Digit Class")

# Plot 2: t-SNE Projection
scatter_t = axes[1].scatter(X_tsne_2d[:, 0], X_tsne_2d[:, 1], c=y_digits, cmap='tab10', s=30, alpha=0.75)
axes[1].set_title("t-SNE (Non-Linear Manifold Learning)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("t-SNE Dimension 1", fontsize=11)
axes[1].set_ylabel("t-SNE Dimension 2", fontsize=11)
plt.colorbar(scatter_t, ax=axes[1], label="Digit Class")

plt.tight_layout()
plt.show()


---
# 6. Principal Component Analysis (PCA)

PCA is an orthogonal linear transformation that maps high-dimensional data onto principal axes maximizing variance:
$$\Sigma v_i = \lambda_i v_i$$

Where $v_i$ are eigenvectors (principal directions) and $\lambda_i$ are eigenvalues (variance captured along $v_i$).

> 🍋 **Layman's Analogy ("Explain Like I'm 5"):**
> Imagine taking a single 2D photograph of a bicycle:
> - If you take the photo from the front, it looks like a skinny pole and you cannot tell what it is!
> - If you take the photo from the side, you see the wheels, frame, handlebars, and pedals in full glory.
> **PCA is the mathematical camera operator** that rotates around your high-dimensional data and finds the exact angle that captures the maximum spread and detail in the fewest snapshots!


### Step 1: Importing

> 💡 *Beginner Tip:* We import only the exact clustering classes, dendrogram tools, and evaluation metrics needed for this recipe.
Imports for PCA and metrics.


In [ ]:
# 1. Importing libraries for Principal Component Analysis (PCA)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error



### Step 2: Loading Dataset

> 💡 *Beginner Tip:* We load the raw data into a DataFrame. Notice there are NO labels for training—the algorithm must discover patterns purely from feature measurements.
Loading Breast Cancer Wisconsin dataset (569 samples, 30 features, Malignant vs. Benign).

[`Breast Cancer Dataset`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)


In [ ]:
# 2. Loading Dataset
cancer_pca = load_breast_cancer(as_frame=True)
df_cancer_pca = cancer_pca.frame

X_pca_raw = df_cancer_pca[cancer_pca.feature_names]
y_pca = df_cancer_pca['target']

print(f"Dataset Shape: {X_pca_raw.shape} (30 continuous features)")
display(X_pca_raw.head(3))


### Step 3: Preprocessing & Mandatory Standardization

> 💡 *Beginner Tip:* Distance-based algorithms (like clustering and PCA) are extremely sensitive to scale! We standardize numbers so huge measurements don't bully smaller ones.
**Why standardization is mandatory for PCA:**
PCA maximizes variance. Unscaled features with large raw magnitudes (e.g. area $> 1000$) will artificially dominate principal components over smaller features (e.g. smoothness $< 0.1$). `StandardScaler` gives all features mean 0 and variance 1.


In [ ]:
# 3. Preprocessing: Mandatory Standardization
scaler_pca = StandardScaler()
X_pca_scaled = scaler_pca.fit_transform(X_pca_raw)
print("All 30 features standardized to zero mean and unit variance.")


### Step 4: Train / Model Setup

> 💡 *Beginner Tip:* We initialize our clustering tool and set our preferences, such as how many clusters ($k$) or what distance threshold to use.
Initializing PCA with all 30 components to examine the full variance spectrum, and 2 components for 2D visualization.


In [ ]:
# 4. Train: Model Initialization
pca_full = PCA(random_state=42)
pca_2d = PCA(n_components=2, random_state=42)
print("PCA models initialized.")


### Step 5: Fit

> 💡 *Beginner Tip:* The computer groups the points together based on closeness, density, or variance.
Computing covariance matrix eigenvalues and eigenvectors.


In [ ]:
# 5. Fit: Model Training
pca_full.fit(X_pca_scaled)
pca_2d.fit(X_pca_scaled)
print("Eigenvalue decomposition complete!")


### Step 6: Predict / Transform

> 💡 *Beginner Tip:* We assign every data point its official cluster tag (0, 1, 2... or -1 for noise).
Projecting 30-dimensional samples onto top 2 principal components.


In [ ]:
# 6. Predict / Transform: Dimensionality Projection
X_projected_2d = pca_2d.transform(X_pca_scaled)
print(f"Projected 2D coordinates shape: {X_projected_2d.shape}")


### Step 7: Evaluation

> 💡 *Beginner Tip:* Since we don't have true answer sheets, we grade clusters using intrinsic geometric quality: Silhouette score measures separation and cohesion.
**Evaluation Metrics:**
- **Explained Variance Ratio:** Individual variance proportion per PC.
- **Cumulative Explained Variance:** Thresholds for retaining 90% and 95% variance.
- **Reconstruction Error (Loss):** MSE between original features and inverse-transformed samples.


In [ ]:
# 7. Evaluation: Variance Diagnostics & Reconstruction Loss
exp_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

n_90 = np.argmax(cum_var >= 0.90) + 1
n_95 = np.argmax(cum_var >= 0.95) + 1

# Reconstruction Loss from 2 components
X_recon_2d = pca_2d.inverse_transform(X_projected_2d)
recon_error = mean_squared_error(X_pca_scaled, X_recon_2d)

print("=" * 60)
print("PCA VARIANCE & RECONSTRUCTION EVALUATION")
print("=" * 60)
print(f"PC1 Variance Explained:                 {exp_var[0]*100:.2f}%")
print(f"PC2 Variance Explained:                 {exp_var[1]*100:.2f}%")
print(f"Top 2 PCs Combined Variance:            {cum_var[1]*100:.2f}%")
print(f"Components required for >= 90% variance:{n_90} of 30")
print(f"Components required for >= 95% variance:{n_95} of 30")
print(f"2D Reconstruction MSE Error:            {recon_error:.4f}")
print("=" * 60)


### Step 8: Visualization

> 💡 *Beginner Tip:* We plot colorful 2D maps, decision boundaries, or tree dendrograms to visually inspect how well the groups separated.
**Visualizations:**
1. **Scree Plot:** Bar chart of individual variance and cumulative curve with 90%/95% thresholds.
2. **2D PCA Projection:** Clear class separation of Malignant vs. Benign.
3. **PCA Loadings Heatmap:** Top original features driving PC1 and PC2.
4. **Reconstruction Error Curve:** Loss vs. number of components.


In [ ]:
# 8. Visualization: Scree Plot, 2D Projection, Loadings Heatmap & Reconstruction Curve
# Plot 1: Scree Plot
plt.figure(figsize=(12, 5))
plt.bar(range(1, 31), exp_var, alpha=0.6, color='#1f77b4', label='Individual Explained Variance')
plt.step(range(1, 31), cum_var, where='mid', color='red', linewidth=2.5, label='Cumulative Explained Variance')
plt.axhline(y=0.90, color='grey', linestyle='--', label='90% Variance Threshold')
plt.axhline(y=0.95, color='black', linestyle=':', label='95% Variance Threshold')
plt.title("PCA Scree Plot: Explained Variance by Principal Components", fontsize=14, fontweight="bold")
plt.xlabel("Principal Component Index", fontsize=11)
plt.ylabel("Explained Variance Ratio", fontsize=11)
plt.xticks(range(1, 31))
plt.legend(loc="center right")
plt.tight_layout()
plt.show()

# Plot 2: 2D Projection
plt.figure(figsize=(9, 5))
scatter_p = plt.scatter(X_projected_2d[:, 0], X_projected_2d[:, 1], c=y_pca, cmap='coolwarm', s=55, alpha=0.85, edgecolors='k')
plt.title(f"2D PCA Projection (Total Variance Captured: {cum_var[1]*100:.1f}%)", fontsize=13, fontweight="bold")
plt.xlabel(f"PC 1 ({exp_var[0]*100:.1f}% Variance)", fontsize=11)
plt.ylabel(f"PC 2 ({exp_var[1]*100:.1f}% Variance)", fontsize=11)
plt.legend(handles=scatter_p.legend_elements()[0], labels=list(cancer_pca.target_names), title="Diagnosis")
plt.tight_layout()
plt.show()

# Plot 3: Loadings Heatmap
loadings = pd.DataFrame(pca_2d.components_.T, columns=['PC1', 'PC2'], index=cancer_pca.feature_names)
top_feats = loadings.abs().sort_values(by='PC1', ascending=False).head(12).index

plt.figure(figsize=(8, 6))
sns.heatmap(loadings.loc[top_feats], annot=True, cmap="vlag", center=0, fmt=".3f", linewidths=0.5)
plt.title("PCA Loadings: Top Contributing Features to PC1 & PC2", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Plot 4: Reconstruction Error Curve
errors = []
for n in range(1, 31):
    p_k = PCA(n_components=n, random_state=42)
    X_t = p_k.fit_transform(X_pca_scaled)
    X_r = p_k.inverse_transform(X_t)
    errors.append(mean_squared_error(X_pca_scaled, X_r))

plt.figure(figsize=(10, 4.5))
plt.plot(range(1, 31), errors, marker='o', color='#d95f02', linewidth=2)
plt.axvline(x=n_95, color='black', linestyle='--', label=f'95% Variance ({n_95} components)')
plt.title("PCA Reconstruction Loss (MSE) vs. Number of Components", fontsize=13, fontweight="bold")
plt.xlabel("Number of Principal Components", fontsize=11)
plt.ylabel("Reconstruction Error (MSE)", fontsize=11)
plt.xticks(range(1, 31, 2))
plt.legend()
plt.tight_layout()
plt.show()


---
## 7. Comparative Summary: Unsupervised Learning Guide

| Topic / Technique | Category | Core Objective | Key Evaluation Metrics | Scaling Required? | Best Applied When |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **K-Means Clustering** | Partitioning Clustering | Minimize within-cluster variance around $K$ centroids | Inertia (WCSS), Silhouette Score, Davies-Bouldin Index | **Yes** (Euclidean distance) | Spherical, convex clusters with known/estimated $K$ |
| **Hierarchical Clustering** | Agglomerative Clustering | Build a nested hierarchy of clusters via linkage criteria | Dendrogram distance, Silhouette Score | **Yes** | Hierarchical taxonomy desired; small to medium datasets |
| **DBSCAN** | Density-Based Clustering | Discover dense regions separated by sparse noise areas | Silhouette Score (non-noise), Noise percentage | **Yes** | Arbitrary non-linear shapes; data with outliers/noise |
| **Association Rules** | Rule / Pattern Mining | Uncover frequent itemsets and co-occurrence rules | Support, Confidence, Lift, Leverage | **No** (Binary transactions) | Market basket analysis, recommendation systems |
| **Truncated SVD** | Linear Dimensionality Reduction | Factorize matrix into top singular components | Explained Variance Ratio, Reconstruction Error | Recommended | Sparse text matrices (LSA), fast linear reduction |
| **t-SNE** | Non-Linear Manifold Learning | Preserve local neighbor probabilities in 2D/3D | KL Divergence | **Yes** | Visualizing complex high-dimensional cluster separation |
| **PCA** | Linear Dimensionality Reduction | Maximize variance along orthogonal principal axes | Explained Variance Ratio, Scree plot, Reconstruction MSE | **Yes** (Mandatory) | Multicollinearity removal, data compression, visualization |
